## Constants and imports

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics
import shap
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from scipy.stats import wasserstein_distance
from scipy.stats import chi2_contingency
from scipy.cluster.hierarchy import linkage, leaves_list



BASE_DIR = "/groups/assafza_group/assafza/"

## get all motif participating unidentified cells

In [ ]:



# base directory containing motif instance files
#txt files with motif annotations
# and csv files with motif hits
MOTIF_BASE_DIR = os.path.join(BASE_DIR, "Omri/melanoma_motif_instances")


unidentified_in_motif_cells = pd.DataFrame(columns=['patient', 'FOV', 'cell','which_motif'])

motif_files = [f for f in os.listdir(MOTIF_BASE_DIR) if f.endswith('.txt')]
for index, motif_file in enumerate(motif_files):
    motif_path = os.path.join(MOTIF_BASE_DIR, motif_file)
    df = pd.read_csv(motif_path.split('.')[0] + '_hits_wide.csv')
    annotation = open(motif_path, 'r').readlines()
    for line in annotation:
        ###############################
        
        """ change to mac \ unidentified to alter the pipeline"""
        
        ###############################
        if 'unidentified' in line:
            cell_num = line.strip()[0]
            #print(cell_num)
            cell_num = ord(cell_num) - ord('A') + 1
            df_temp = df[['patient', 'FOV', f'cell{cell_num}']]
            df_temp = df_temp.rename(columns={f'cell{cell_num}': 'cell'})
            df_temp['which_motif'] = motif_file.split('.')[0]
            #display(df_temp)
            unidentified_in_motif_cells = pd.concat([unidentified_in_motif_cells, df_temp])
            
unidentified_in_motif_cells['label'] = 1
unidentified_in_motif_cells.head(20)
            

In [ ]:
protein_expression_dataset_path = os.path.join(BASE_DIR, "Datasets/Melanoma/cleaned_expression_with_both_classification_prob_spatial_27_09_23.csv")
protein_expression_dataset = pd.read_csv(protein_expression_dataset_path)
protein_expression_dataset.head()

In [ ]:
wanted_columns = ['patient number','fov','centroid-0' ,'centroid-1','CD11c', 'CD14', 'CD163', 'CD20', 'CD206', 'CD21', 'CD3', 'CD31', 'CD4',
       'CD45', 'CD45RA', 'CD45RO', 'CD56', 'CD68', 'CD69', 'CD8', 'COL1A1',
       'DC-SIGN', 'Foxp3', 'Granzyme B', 'HLA-DR-DP-DQ', 'HLA-class-1-A-B-C',
       'IDO-1', 'Ki67', 'LAG-3', 'MECA-79', 'MelanA', 'PD-1',
       'S100A9-Calprotectin', 'SMA', 'SOX10', 'TCF1TCF7', 'TIM-3', 'Tox-Tox2',
       'anti-Biotin', 'dsDNA']
protein_expression_dataset = protein_expression_dataset[wanted_columns]

In [ ]:
melanoma_data_path = os.path.join(BASE_DIR, "Datasets/Melanoma/Melanona_MIBI_TOF_18_7_2024.csv")
melanoma_cells = pd.read_csv(melanoma_data_path)
melanoma_cells.head()

### Round centroid values so it fits with the protein expression data

In [ ]:
melanoma_cells['centroid-0'] = melanoma_cells['centroid-0'].apply(lambda x: int(x)).astype(float)
melanoma_cells['centroid-1'] = melanoma_cells['centroid-1'].apply(lambda x: int(x)).astype(float)

### fit column names for future merge

In [ ]:
unidentified_in_motif_cells.rename(columns={'FOV':'fov','cell':'cell_id','patient':'patient number'}, inplace=True)

## Merge

### change to merge oonly on FOV and change patient number 72 on the protein expression dataset to the matching patient number's FOVs on the melanoma cells dataset  which is 92

In [ ]:
melanoma_protein = pd.merge(melanoma_cells, protein_expression_dataset, on = ['patient number','fov','centroid-0','centroid-1'], how = 'inner')

In [ ]:
melanoma_protein.drop(columns=['Unnamed: 0'], inplace=True)

In [ ]:
melanoma_protein = melanoma_protein[melanoma_protein['Group'].isin(['NN','NP'])]

In [ ]:
classification_ready_dataset = melanoma_protein.drop(columns=['cell_id'])

In [ ]:
unidentified_in_motif_cells

In [ ]:
unidentified_wanted_columns = ['patient number','fov','cell_id','label', 'which_motif'] 
unidentified_in_motif_cells = unidentified_in_motif_cells[unidentified_wanted_columns]

In [ ]:
unidentified_in_motif_cells.head()

In [ ]:
melanoma_protein = pd.merge(melanoma_protein, unidentified_in_motif_cells, on = ['patient number','fov','cell_id'], how = 'left', suffixes=('_all', '_unidentified'))

In [ ]:
melanoma_protein['meta_pred'].unique()

In [ ]:
melanoma_protein['label'] = melanoma_protein['label'].fillna(0).astype(int)
melanoma_protein['which_motif'] = melanoma_protein['which_motif'].fillna(0).astype(int)

In [ ]:
################

"""change to mac \ unidentified to alter the pipeline"""

###############

melanoma_protein_unidentified = melanoma_protein[melanoma_protein['pred']=='Unidentified']
melanoma_protein_unidentified.shape

In [ ]:
classifier_unwanted_columns = ['centroid-0','centroid-1','patient number','fov','cell_id','pred','Group']
df = melanoma_protein_unidentified.drop(columns=classifier_unwanted_columns)

# Classification

In [ ]:


# prepare features and target (use numeric columns only)
X = df.select_dtypes(include=[np.number]).drop(columns=['label'])
y = df['label'].astype(int)

cell_num = 42069  # replace with the actual cell number used in previous cells
# train/test split (use provided cell_num as random_state)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=cell_num, stratify=y
)

# fit random forest
rf = RandomForestClassifier(n_estimators=200, random_state=cell_num, n_jobs=-1)
rf.fit(X_train, y_train)

# evaluation
y_pred = rf.predict(X_test)
print("Accuracy:", metrics.accuracy_score(y_test, y_pred))
print(metrics.classification_report(y_test, y_pred))

# feature importance
fi = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top features:\n", fi.head(20))

# SHAP explanation (sample if test set is large)
shap_sample = X_test.sample(n=min(len(X_test), 2000), random_state=cell_num)
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(shap_sample)

# For binary classification shap_values is a list [neg_class, pos_class]; show pos class
if isinstance(shap_values, list) and len(shap_values) == 2:
    sv = shap_values[1]
else:
    sv = shap_values

# summary plot
plt.figure(figsize=(8,6))
shap.summary_plot(sv, shap_sample, show=True)

# Clustering

In [ ]:


# use df for clustering (numeric features only, exclude label)
X = df.select_dtypes(include=[np.number]).drop(columns=['label','which_motif'], errors='ignore')
X_index = X.index

# scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# sample for model selection (max 5000)
sample_size = min(len(X_scaled), 5000)
rng = np.random.RandomState(42069)
sample_idx = rng.choice(len(X_scaled), size=sample_size, replace=False)
X_sample = X_scaled[sample_idx]

# evaluate range of k (include 15)
ks = list(range(2, 21))
if 15 not in ks:
    ks.append(15)
ks = sorted(set(ks))

sil_scores = []
inertias = []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42069, n_init=10)
    km.fit(X_sample)
    sil = silhouette_score(X_sample, km.labels_)
    sil_scores.append(sil)
    inertias.append(km.inertia_)
    print(f"k={k}: silhouette={sil:.4f}, inertia={km.inertia_:.1f}")

# plot results
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(ks, sil_scores, marker='o')
plt.xlabel('k'); plt.ylabel('Silhouette score'); plt.title('Silhouette vs k')
plt.subplot(1,2,2)
plt.plot(ks, inertias, marker='o')
plt.xlabel('k'); plt.ylabel('Inertia'); plt.title('Inertia vs k')
plt.tight_layout()
plt.show()

# pick best k by silhouette
best_k = ks[int(np.argmax(sil_scores))]
print("Best k (by silhouette):", best_k)

# fit KMeans on full scaled data for best_k and for k=15
km_best = KMeans(n_clusters=best_k, random_state=42069, n_init=10).fit(X_scaled)
km_8 = KMeans(n_clusters=8, random_state=42069, n_init=10).fit(X_scaled)
km_15   = KMeans(n_clusters=15, random_state=42069, n_init=10).fit(X_scaled)

# attach cluster labels to df (align by index)
df.loc[X_index, 'cluster_best'] = km_best.labels_
df.loc[X_index, 'cluster_8'] = km_8.labels_
df.loc[X_index, 'cluster_15'] = km_15.labels_

def hierarchical_cluster_order(df_, cluster_col, feature_cols=None):
    if cluster_col not in df_.columns:
        return []
    if feature_cols is None:
        excluded = {cluster_col, 'label', 'which_motif', 'cluster_best', 'cluster_8', 'cluster_15', 'class'}
        feature_cols = [c for c in df_.select_dtypes(include=[np.number]).columns if c not in excluded]
    feature_cols = [c for c in feature_cols if c in df_.columns]
    if not feature_cols:
        return sorted(df_[cluster_col].dropna().astype(int).unique().tolist())

    tmp = df_[[cluster_col] + feature_cols].dropna(subset=[cluster_col]).copy()
    if tmp.empty:
        return []
    tmp[cluster_col] = tmp[cluster_col].astype(int)
    centroids = tmp.groupby(cluster_col)[feature_cols].mean()

    if centroids.shape[0] <= 2:
        return centroids.index.tolist()

    z = linkage(centroids.values, method='average', metric='euclidean')
    leaf_order = leaves_list(z)
    return centroids.index.to_numpy()[leaf_order].tolist()

# show how the labeled cells are distributed between clusters
def show_distribution(df_, cluster_col):
    ct = df_.groupby([cluster_col, 'which_motif']).size().unstack(fill_value=0)
    pct = (ct.div(ct.sum(axis=1), axis=0)*100).round(2)
    print(f"\nDistribution by {cluster_col} (counts):")
    print(ct)
    print(f"\nDistribution by {cluster_col} (percentages per cluster):")
    print(pct)

show_distribution(df, 'cluster_best')
show_distribution(df, 'cluster_8')
show_distribution(df, 'cluster_15')

In [ ]:
show_distribution(df, 'cluster_best')
show_distribution(df, 'cluster_15')

## Chi-square test for distribution differences between general unidentified cells and motif participating unidentified cells

In [ ]:
# Chi-square tests comparing distribution of 'label' (in-motif unidentified = 1)
# across clusters. Uses both cluster_best and cluster_15 if present.

# df is expected to contain the cluster columns and 'label' (0/1)
for col in ['cluster_best', 'cluster_15']:
    print('---')
    if col not in df.columns:

        continue
    # build contingency table: rows=cluster, cols=label (0/1)
    ct = pd.crosstab(df[col].fillna(-1).astype(int), df['which_motif'])

    print(ct)
    # need at least 2x2 to run chi-square
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        print('Not enough categories to run chi-square test; need at least 2x2.')
        continue
    try:
        chi2, p, dof, expected = chi2_contingency(ct)

        exp_df = pd.DataFrame(expected, index=ct.index, columns=ct.columns)
        print('Expected frequencies:')
        print(exp_df.round(2))
        print(f'Chi-square test results for {col}: p_value : {p} , chi : {chi2}')
    except Exception as e:
        print('Error running chi-square test for', col, ':', e)

# End of chi-square checks

## Wassertein distance between general unidentified cells and motif participating unidentified cells

In [ ]:
# Wasserstein distance comparing cluster assignment distributions for label==1 vs label==0


for col in ['cluster_best', 'cluster_15']:
    print('---')
    if col not in df.columns:
        print(f"Column {col} not found in dataframe; skipping.")
        continue
    # extract cluster indices for the two groups
    a = df.loc[df['label'] == 1, col].dropna().astype(int).values
    b = df.loc[df['label'] == 0, col].dropna().astype(int).values
    print(f"Column {col}: n_in_motif={len(a)}, n_other={len(b)}")
    if len(a) == 0 or len(b) == 0:
        print('Not enough samples to compute Wasserstein distance; need non-empty groups.')
        continue
    # compute Wasserstein distance on the cluster index values (1D)
    wd = wasserstein_distance(a, b)
    print(f'Wasserstein distance (cluster index space) between in-motif and other: {wd:.4f}')

    # show per-cluster proportions for reference
    all_vals = np.union1d(np.unique(a), np.unique(b))
    minc = int(all_vals.min())
    maxc = int(all_vals.max())
    bins = np.arange(minc, maxc + 1)
    pa = np.array([np.sum(a == c) for c in bins]) / len(a)
    pb = np.array([np.sum(b == c) for c in bins]) / len(b)
    hist_df = pd.DataFrame({'cluster': bins, 'p_in_motif': pa, 'p_other': pb})
    print('Per-cluster proportions:')
    print(hist_df.round(4).to_string(index=False))

# End Wasserstein checks

## Plots for unidentified cells in motif distribution over clusters for each motif

In [ ]:
# Faceted radar plots per which_motif for each cluster column
cluster_cols = [c for c in ['cluster_best', 'cluster_15', 'cluster_8'] if c in df.columns]
if not cluster_cols:
    raise RuntimeError("No cluster columns ('cluster_best' or 'cluster_15') found in df.")

# motifs to iterate (exclude 0 if you don't want the 'none' group)
motifs = sorted(df['which_motif'].dropna().astype(int).unique())
# optionally exclude 0 (unassigned)
motifs = [m for m in motifs if m != 0]
print(len(motifs), "motifs found for plotting.")

for col in cluster_cols:
    clusters = hierarchical_cluster_order(df, col, feature_cols=X.columns.tolist())
    n_clusters = len(clusters)
    if n_clusters == 0:
        print(f"No clusters found for {col}; skipping.")
        continue

    # precompute global "other" distribution (label==0) once for consistent comparison
    counts_out_global = np.array([((df[col].astype(int) == c) & (df['label'] == 0)).sum() for c in clusters], dtype=float)
    total_out = counts_out_global.sum()
    prop_out_global = counts_out_global / total_out if total_out > 0 else np.zeros_like(counts_out_global)

    # compute a common y-scale across motifs
    max_prop_across = 0.0
    per_motif_props = {}
    per_motif_counts = {}
    for m in motifs:
        counts_in = np.array([((df[col].astype(int) == c) & (df['which_motif'] == m) & (df['label'] == 1)).sum() for c in clusters], dtype=float)
        total_in = counts_in.sum()
        prop_in = counts_in / total_in if total_in > 0 else np.zeros_like(counts_in)
        per_motif_props[m] = (prop_in, prop_out_global)
        per_motif_counts[m] = (counts_in.astype(int), counts_out_global.astype(int))
        max_prop_across = max(max_prop_across, prop_in.max(), prop_out_global.max())

    # layout
    n_motifs = len(motifs)
    ncols = 4
    nrows = int(np.ceil(n_motifs / ncols))
    fig = plt.figure(figsize=(4 * ncols, 4 * nrows))
    plt.suptitle(f'Cluster distribution by label — faceted by which_motif ({col})', fontsize=16)

    for i, m in enumerate(motifs):
        print(f"Plotting motif {m} ({i+1}/{n_motifs})...")
        prop_in, prop_out = per_motif_props[m]
        angles = np.linspace(0, 2 * np.pi, n_clusters, endpoint=False)
        angles = np.concatenate([angles, [angles[0]]])
        prop_in_closed = np.concatenate([prop_in, [prop_in[0]]])
        prop_out_closed = np.concatenate([prop_out, [prop_out[0]]])

        ax = fig.add_subplot(nrows, ncols, i + 1, polar=True)
        ax.plot(angles, prop_in_closed, color='C0', linewidth=2, label='in-motif')
        ax.fill_between(angles, 0, prop_in_closed, color='C0', alpha=0.25)
        ax.plot(angles, prop_out_closed, color='C1', linewidth=2, label='other')
        ax.fill_between(angles, 0, prop_out_closed, color='C1', alpha=0.25)

        ax.set_xticks(angles[:-1])
        ax.set_xticklabels([str(c) for c in clusters], fontsize=8)
        # y ticks
        y_max = max_prop_across if max_prop_across > 0 else 0.1
        ax.set_yticks(np.linspace(0, y_max, 4))
        ax.set_ylim(0, y_max * 1.05)

        counts_in, counts_out = per_motif_counts[m]
        ax.set_title(f'motif {m}\n(n_in={int(counts_in.sum())}, n_other={int(counts_out.sum())})', fontsize=10)

    # hide any empty subplots
    # for j in range(n_motifs, nrows * ncols):
    #     fig.delaxes(fig.axes[-1])

    # single legend
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper right')
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

    # print table of proportions for each motif (optional)
    # for m in motifs:
    #     prop_in, prop_out = per_motif_props[m]
    #     counts_in, counts_out = per_motif_counts[m]
    #     pct_df = pd.DataFrame({
    #         'cluster': clusters,
    #         'p_in_motif': np.round(prop_in, 4),
    #         'p_other'   : np.round(prop_out, 4),
    #         'count_in'  : counts_in,
    #         'count_out' : counts_out,
    #     })
    #     print(f"\nPer-cluster proportions for {col}, which_motif={m}:")
    #     display(pct_df)

In [ ]:

# Heatmaps: cluster distribution per motif
# Uses: per_motif_props, per_motif_counts, clusters, motifs

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Only show cluster_15
col = 'cluster_15'
if col in df.columns:
    clusters = hierarchical_cluster_order(df, col, feature_cols=X.columns.tolist())
    
    # precompute global "other" distribution
    counts_out_global = np.array([((df[col].astype(int) == c) & (df['label'] == 0)).sum() for c in clusters], dtype=float)
    total_out = counts_out_global.sum()
    prop_out_global = counts_out_global / total_out if total_out > 0 else np.zeros_like(counts_out_global)

    # build matrix: rows = motifs + 1 (for general population), cols = clusters, values = proportions
    matrix_in = np.zeros((len(motifs) + 1, len(clusters)))
    
    for i, m in enumerate(motifs):
        counts_in = np.array([((df[col].astype(int) == c) & (df['which_motif'] == m) & (df['label'] == 1)).sum() for c in clusters], dtype=float)
        total_in = counts_in.sum()
        prop_in = counts_in / total_in if total_in > 0 else np.zeros_like(counts_in)
        matrix_in[i, :] = prop_in
    
    # add general population row (prop_out_global)
    matrix_in[-1, :] = prop_out_global
    
    # create heatmap
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # motif labels + "General population" for last row
    motif_labels = [str(m) for m in motifs] + ['General population']
    
    sns.heatmap(matrix_in, annot=False, cmap='Blues', ax=ax,
                xticklabels=[str(c) for c in clusters], yticklabels=motif_labels,
                cbar_kws={'label': 'Proportion'})
    #ax.set_title(f'Cluster distribution — in-motif cells and general population (cluster_15)', fontsize=12)
    ax.set_xlabel('Cluster', fontsize=11)
    ax.set_ylabel('Motif / Population', fontsize=11)
    
    plt.tight_layout()
    plt.show()


In [ ]:

# Heatmap: (in-motif proportion - general population proportion) per cluster, no general population row.
# Plus bar plot of general population % per cluster.

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

col = 'cluster_15'
if col in df.columns:
    clusters = hierarchical_cluster_order(df, col, feature_cols=X.columns.tolist())

    # general population distribution
    counts_out_global = np.array(
        [((df[col].astype(int) == c) & (df['label'] == 0)).sum() for c in clusters], dtype=float
    )
    total_out = counts_out_global.sum()
    prop_out_global = counts_out_global / total_out if total_out > 0 else np.zeros_like(counts_out_global)

    # build difference matrix: rows = motifs, cols = clusters
    matrix_diff = np.zeros((len(motifs), len(clusters)))
    for i, m in enumerate(motifs):
        counts_in = np.array(
            [((df[col].astype(int) == c) & (df['which_motif'] == m) & (df['label'] == 1)).sum()
             for c in clusters], dtype=float
        )
        total_in = counts_in.sum()
        prop_in = counts_in / total_in if total_in > 0 else np.zeros_like(counts_in)
        matrix_diff[i, :] = prop_in - prop_out_global

    vmax = np.abs(matrix_diff).max()

    fig, axes = plt.subplots(
        2, 1,
        figsize=(12, max(8, len(motifs) * 0.8) + 4),
        gridspec_kw={'height_ratios': [len(motifs), 2.5]}
    )

    # --- top: difference heatmap ---
    ax_heat = axes[0]
    sns.heatmap(
        matrix_diff, annot=False, cmap='RdBu_r', center=0,
        ax=ax_heat, vmin=-vmax, vmax=vmax,
        xticklabels=[str(c) for c in clusters],
        yticklabels=[str(m) for m in motifs],
        cbar_kws={'label': 'Proportion difference\n(in-motif − general population)'}
    )
    ax_heat.set_xlabel('Cluster', fontsize=11)
    ax_heat.set_ylabel('Motif', fontsize=11)
    ax_heat.set_title(
        f'In-motif proportion minus general population proportion per cluster ({col})', fontsize=12
    )

    # --- bottom: bar plot of general population % per cluster ---
    ax_bar = axes[1]
    pct_out = prop_out_global * 100
    x_pos = np.arange(len(clusters))
    ax_bar.bar(x_pos, pct_out, color='steelblue', alpha=0.8, edgecolor='black', linewidth=0.6)
    ax_bar.set_xticks(x_pos)
    ax_bar.set_xticklabels([str(c) for c in clusters])
    ax_bar.set_xlabel('Cluster', fontsize=11)
    ax_bar.set_ylabel('Percentage (%)', fontsize=11)
    ax_bar.set_title('General population distribution per cluster', fontsize=12)
    ax_bar.grid(axis='y', alpha=0.3)
    for xi, val in zip(x_pos, pct_out):
        ax_bar.text(xi, val + 0.1, f'{val:.1f}%', ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    plt.show()


In [ ]:
NN_motif_ids = [21276842380, 1672625548, 1677351212, 20202011980, 20748339660, 17763644908, 11561745772, 11583232364]
NP_motif_ids = [12110603532, 2454267148, 2961208588, 15901249932, 1639021836, 4880694732, 16417793484, 4071751116, 3522784716, 11075202540, 6191483428857422447]

In [ ]:
col = 'cluster_15'
if col in df.columns:
    clusters = hierarchical_cluster_order(df, col, feature_cols=X.columns.tolist())
    motif_vals = pd.to_numeric(df['which_motif'], errors='coerce')

    # general population distribution
    counts_out_global = np.array(
        [((df[col].astype(int) == c) & (df['label'] == 0)).sum() for c in clusters], dtype=float
    )
    total_out = counts_out_global.sum()
    prop_out_global = counts_out_global / total_out if total_out > 0 else np.zeros_like(counts_out_global)

    # sort motifs: NN first, then NP, then any remaining
    nn_motifs = [m for m in motifs if m in NN_motif_ids]
    np_motifs = [m for m in motifs if m in NP_motif_ids]
    other_motifs = [m for m in motifs if m not in NN_motif_ids and m not in NP_motif_ids]
    sorted_motifs = nn_motifs + np_motifs + other_motifs

    # build division matrix: rows = motifs, cols = clusters (motif / general)
    matrix_ratio = np.full((len(sorted_motifs), len(clusters)), np.nan, dtype=float)
    for i, m in enumerate(sorted_motifs):
        counts_in = np.array(
            [((df[col].astype(int) == c) & (motif_vals == m) & (df['label'] == 1)).sum()
             for c in clusters], dtype=float
        )
        total_in = counts_in.sum()
        prop_in = counts_in / total_in if total_in > 0 else np.zeros_like(counts_in)

        matrix_ratio[i, :] = np.divide(
            prop_in,
            prop_out_global,
            out=np.full_like(prop_in, np.nan),
            where=prop_out_global > 0
        )

    cell_w = 0.7
    cell_h = 0.5
    cbar_w = 0.1
    fig_w = len(clusters) * cell_w + cbar_w
    fig_h = len(sorted_motifs) * cell_h

    fig, ax_heat = plt.subplots(figsize=(fig_w, fig_h))

    sns.heatmap(
        matrix_ratio, annot=False, cmap='viridis',
        ax=ax_heat,
        xticklabels=False, yticklabels=False,
        cbar=True,
        cbar_kws={'location': 'left', 'label': 'motif / general'}
    )
    ax_heat.set_xticks([])  # remove x ticks
    ax_heat.set_xlabel('cluster')
    ax_heat.set_ylabel('motif')
    ax_heat.set_title('Cluster distribution ratio (in-motif / general population) per motif', fontsize=12)

    plt.tight_layout()
    plt.show()


In [ ]:

# Two companion heatmaps with numbers, same viridis scale:
#  1. Raw in-motif proportion per motif × cluster (before dividing by general population)
#  2. General population proportion row duplicated for every motif

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

col = 'cluster_15'
if col in df.columns:
    clusters = hierarchical_cluster_order(df, col, feature_cols=X.columns.tolist())
    motif_vals = pd.to_numeric(df['which_motif'], errors='coerce')

    # ── general population ────────────────────────────────────────────────────
    counts_out_global = np.array(
        [((df[col].astype(int) == c) & (df['label'] == 0)).sum() for c in clusters], dtype=float
    )
    total_out = counts_out_global.sum()
    prop_out_global = counts_out_global / total_out if total_out > 0 else np.zeros_like(counts_out_global)

    # ── motif ordering (same as cell 39) ─────────────────────────────────────
    nn_m   = [m for m in motifs if m in NN_motif_ids]
    np_m   = [m for m in motifs if m in NP_motif_ids]
    oth_m  = [m for m in motifs if m not in NN_motif_ids and m not in NP_motif_ids]
    s_motifs = nn_m + np_m + oth_m
    n_motifs_s = len(s_motifs)

    # ── build raw proportion matrix ───────────────────────────────────────────
    matrix_prop = np.zeros((n_motifs_s, len(clusters)), dtype=float)
    for i, m in enumerate(s_motifs):
        counts_in = np.array(
            [((df[col].astype(int) == c) & (motif_vals == m) & (df['label'] == 1)).sum()
             for c in clusters], dtype=float
        )
        total_in = counts_in.sum()
        matrix_prop[i, :] = counts_in / total_in if total_in > 0 else 0.0

    # ── build tiled general-population matrix ─────────────────────────────────
    matrix_genpop = np.tile(prop_out_global, (n_motifs_s, 1))   # (n_motifs, n_clusters)

    # ── shared colour scale ───────────────────────────────────────────────────
    all_vals = np.concatenate([matrix_prop.ravel(), matrix_genpop.ravel()])
    vmin_shared = float(np.nanmin(all_vals))
    vmax_shared = float(np.nanmax(all_vals))

    # ── figure sizing (same cell-size logic as cell 39) ───────────────────────
    cell_w, cell_h, cbar_w = 0.7, 0.5, 0.1
    fig_w  = len(clusters) * cell_w + cbar_w + 1
    fig_h  = n_motifs_s * cell_h + 1

    motif_labels = [str(m) for m in s_motifs]
    cluster_labels_str = [str(c) for c in clusters]

    fig, axes = plt.subplots(1, 2, figsize=(fig_w * 2 + 0.5, fig_h),
                             gridspec_kw={'wspace': 0.4})

    # ── heatmap 1: raw in-motif proportion ────────────────────────────────────
    sns.heatmap(
        matrix_prop,
        annot=True, fmt='.3f', annot_kws={'size': 7},
        cmap='viridis', vmin=vmin_shared, vmax=vmax_shared,
        ax=axes[0],
        xticklabels=cluster_labels_str,
        yticklabels=motif_labels,
        cbar_kws={'label': 'proportion'},
        linewidths=0.2, linecolor='gray'
    )
    axes[0].set_title('In-motif cluster distribution\n(proportion per motif)', fontsize=11)
    axes[0].set_xlabel('cluster', fontsize=10)
    axes[0].set_ylabel('motif', fontsize=10)
    axes[0].tick_params(axis='x', rotation=0, labelsize=8)
    axes[0].tick_params(axis='y', rotation=0, labelsize=7)

    # ── heatmap 2: general population — tiled ────────────────────────────────
    sns.heatmap(
        matrix_genpop,
        annot=True, fmt='.3f', annot_kws={'size': 7},
        cmap='viridis', vmin=vmin_shared, vmax=vmax_shared,
        ax=axes[1],
        xticklabels=cluster_labels_str,
        yticklabels=motif_labels,
        cbar_kws={'label': 'proportion'},
        linewidths=0.2, linecolor='gray'
    )
    axes[1].set_title('General population cluster distribution\n(same row repeated per motif)', fontsize=11)
    axes[1].set_xlabel('cluster', fontsize=10)
    #axes[1].set_ylabel('motif', fontsize=10)
    #axes[1].tick_params(axis='x', rotation=0, labelsize=8)
    #axes[1].tick_params(axis='y', rotation=0, labelsize=7)

    plt.suptitle(f'Cluster proportions — {col}  |  shared colour scale [{vmin_shared:.3f}, {vmax_shared:.3f}]',
                 fontsize=12, y=1.01)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', UserWarning)
        plt.tight_layout()
    plt.show()


In [ ]:

# Log2 enrichment heatmaps: log2(in-motif / other) per cluster and motif
# Uses: per_motif_props, per_motif_counts, clusters, motifs

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

for col in cluster_cols:
    clusters = hierarchical_cluster_order(df, col, feature_cols=X.columns.tolist())
    
    # precompute global "other" distribution
    counts_out_global = np.array([((df[col].astype(int) == c) & (df['label'] == 0)).sum() for c in clusters], dtype=float)
    total_out = counts_out_global.sum()
    prop_out_global = counts_out_global / total_out if total_out > 0 else np.zeros_like(counts_out_global)

    # build matrix: rows = motifs, cols = clusters, values = log2 enrichment
    matrix_log2 = np.zeros((len(motifs), len(clusters)))
    
    pseudocount = 1e-6
    for i, m in enumerate(motifs):
        counts_in = np.array([((df[col].astype(int) == c) & (df['which_motif'] == m) & (df['label'] == 1)).sum() for c in clusters], dtype=float)
        total_in = counts_in.sum()
        prop_in = counts_in / total_in if total_in > 0 else np.zeros_like(counts_in)
        log2_enrichment = np.log2((prop_in + pseudocount) / (prop_out_global + pseudocount))
        matrix_log2[i, :] = log2_enrichment
    
    # determine color scale (symmetric around 0)
    vmax = np.abs(matrix_log2).max()
    
    # create heatmap
    fig, ax = plt.subplots(figsize=(10, 8))
    
    sns.heatmap(matrix_log2, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                ax=ax, vmin=-vmax, vmax=vmax,
                xticklabels=[str(c) for c in clusters], yticklabels=[str(m) for m in motifs],
                cbar_kws={'label': 'Log2 enrichment'})
    
    ax.set_title(f'Log2 enrichment heatmap (in-motif / other) — {col}', fontsize=14)
    ax.set_xlabel('Cluster', fontsize=12)
    ax.set_ylabel('Motif', fontsize=12)
    
    plt.tight_layout()
    plt.show()


## Cluster visualization

In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.preprocessing import StandardScaler

# t-SNE visualization of clusters (new cell)
# re-use existing variables where possible (df, X_scaled, cluster columns)
import matplotlib.pyplot as plt

# choose cluster column to visualize
cluster_cols = [c for c in ['cluster_best', 'cluster_15','cluster_8'] if c in df.columns]
if not cluster_cols:
    raise RuntimeError("No cluster columns ('cluster_best' or 'cluster_15') found in df.")
cluster_col = cluster_cols[1]

# features (numeric only) - avoid dropping features that don't exist
X = df.select_dtypes(include=[np.number]).drop(columns=['label','which_motif'], errors='ignore')

# use existing X_scaled if available, otherwise scale here
if 'X_scaled' in globals() and X.shape[1] == X_scaled.shape[1]:
    X_full = X_scaled
else:
    scaler_tsne = StandardScaler()
    X_full = scaler_tsne.fit_transform(X)

# sample for t-SNE (t-SNE on many points is slow) - stratify by cluster to preserve distribution
n_sample = 5000
n_total = X_full.shape[0]
if n_total > n_sample:
    cluster_labels = df[cluster_col].fillna(-1).astype(int).values
    idx_all = np.arange(n_total)
    _, idx_sample = train_test_split(
        idx_all,
        test_size=n_sample / n_total,
        stratify=cluster_labels,
        random_state=42069
    )
else:
    idx_sample = np.arange(n_total)

X_sample = X_full[idx_sample]
cluster_sample = df.iloc[idx_sample][cluster_col].fillna(-1).astype(int).values
label_sample = df.iloc[idx_sample]['label'].values

# PCA reduction before t-SNE to speed up (common practice)
pca_n = min(50, X_sample.shape[1])
pca = PCA(n_components=pca_n, random_state=42069)
X_pca = pca.fit_transform(X_sample)

# run t-SNE
tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42069, init='pca', learning_rate='auto')
tsne_emb = tsne.fit_transform(X_pca)

# plotting
unique_clusters = np.unique(cluster_sample)
n_clusters = len(unique_clusters)
cmap_clusters = plt.cm.get_cmap('tab20', max(1, n_clusters))

plt.figure(figsize=(14,6))

# plot colored by cluster
ax = plt.subplot(1,2,1)
sc = ax.scatter(tsne_emb[:,0], tsne_emb[:,1], c=cluster_sample, cmap=cmap_clusters, s=8, alpha=0.8)
ax.set_title(f"t-SNE colored by {cluster_col} (n={len(idx_sample)})")
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
# colorbar with cluster ticks
cb = plt.colorbar(sc, ax=ax, ticks=unique_clusters)
cb.set_label('cluster id')

# plot colored by label (in-motif vs other)
ax2 = plt.subplot(1,2,2)
sc2 = ax2.scatter(tsne_emb[:,0], tsne_emb[:,1], c=label_sample, cmap='coolwarm', s=8, alpha=0.8)
ax2.set_title('t-SNE colored by label (in-motif=1)')
ax2.set_xlabel('t-SNE 1'); ax2.set_ylabel('t-SNE 2')
cb2 = plt.colorbar(sc2, ax=ax2, ticks=[0,1])
cb2.set_ticklabels(['other (0)','in-motif (1)'])

plt.tight_layout()
plt.show()

In [ ]:
import math
# Radar plots of cluster centroids (axes = features) with per-feature min-max scaling
# Uses existing variables: X (numeric features dataframe), scaler (StandardScaler), km_best, km_15 (KMeans)

# choose which KMeans objects are available
cluster_models = {}
if 'km_best' in globals():
    cluster_models['cluster_best'] = km_best
if 'km_15' in globals():
    cluster_models['cluster_15'] = km_15
if 'km_8' in globals():
    cluster_models['cluster_8'] = km_8
if not cluster_models:
    raise RuntimeError("No KMeans models (km_best or km_15) found in the notebook namespace.")

# features (order used when clustering)
if 'X' in globals() and hasattr(X, "columns"):
    feat_names = list(X.columns)
else:
    # try to rebuild numeric features from df
    feat_names = list(df.select_dtypes(include=[np.number]).drop(columns=['label','which_motif'], errors='ignore').columns)

# reference data for min/max per feature: prefer protein_expression_dataset if available (original scale),
# otherwise fall back to X (unscaled) or df numeric features.
if 'protein_expression_dataset' in globals():
    ref_df = protein_expression_dataset.copy()
elif 'X' in globals() and hasattr(X, "columns"):
    ref_df = X.copy()
else:
    ref_df = df.select_dtypes(include=[np.number]).drop(columns=['label','which_motif'], errors='ignore')

# align feat_names with ref_df columns (drop features not present)
feat_names = [f for f in feat_names if f in ref_df.columns]
if len(feat_names) == 0:
    raise RuntimeError("No feature columns found for plotting after alignment.")

# compute per-feature min/max for min-max scaling (original scale)
min_vals = ref_df[feat_names].min().values.astype(float)
max_vals = ref_df[feat_names].max().values.astype(float)
# avoid divide-by-zero
denom = max_vals - min_vals
denom[denom == 0] = 1.0

# helper to get centroids in original scale then min-max scale to [0,1]
def get_centroids_scaled(km):
    cent = km.cluster_centers_  # in scaled space used by clustering (likely StandardScaler)
    # try to inverse-transform with scaler to original scale
    if 'scaler' in globals() and hasattr(scaler, "inverse_transform"):
        try:
            cent_orig = scaler.inverse_transform(cent)
        except Exception:
            cent_orig = cent
    else:
        cent_orig = cent
    # align number of features between centroids and feat_names
    if cent_orig.shape[1] != len(feat_names):
        # if mismatch, try to use as many features as possible
        n_use = min(cent_orig.shape[1], len(feat_names))
        cent_orig = cent_orig[:, :n_use]
        use_min = min_vals[:n_use]
        use_denom = denom[:n_use]
        scaled = (cent_orig - use_min) / use_denom
        return scaled, feat_names[:n_use]
    scaled = (cent_orig - min_vals) / denom
    return scaled, feat_names

for name, km in cluster_models.items():
    centroids_scaled, plot_feat_names = get_centroids_scaled(km)
    n_clusters = centroids_scaled.shape[0]
    n_feats = centroids_scaled.shape[1]

    # ensure plotting range 0..1
    vmin = 0.0
    vmax = 1.0

    # subplot grid
    ncols = 4
    nrows = math.ceil(n_clusters / ncols)
    fig = plt.figure(figsize=(4 * ncols, 3.5 * nrows))
    plt.suptitle(f'Cluster centroids (min-max scaled per feature) — {name}', fontsize=14)

    # angles for radar
    angles = np.linspace(0, 2 * np.pi, n_feats, endpoint=False)
    angles_closed = np.concatenate([angles, [angles[0]]])

    for i in range(n_clusters):
        values = centroids_scaled[i]
        values_closed = np.concatenate([values, [values[0]]])

        ax = fig.add_subplot(nrows, ncols, i + 1, polar=True)
        ax.plot(angles_closed, values_closed, color='C0', linewidth=1.5)
        ax.fill_between(angles_closed, 0, values_closed, color='C0', alpha=0.2)

        ax.set_xticks(angles)
        # shorten feature labels if too long
        ax.set_xticklabels([f if len(f) <= 12 else f[:11] + '…' for f in plot_feat_names], fontsize=7)
        # y limits and ticks on 0..1
        ax.set_ylim(vmin, vmax)
        yticks = np.linspace(vmin, vmax, 4)
        ax.set_yticks(yticks)
        ax.set_yticklabels([f"{t:.2f}" for t in yticks], fontsize=6)
        ax.set_title(f'cluster {i}', fontsize=9)

    # remove empty axes if any
    # for j in range(n_clusters, nrows * ncols):
    #     fig.delaxes(fig.axes[-1])

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

In [ ]:
import numpy as np

# Dot plot of cluster centroids for each feature
# Uses existing cluster_models dict (km_best, km_15) and get_centroids_scaled function

import matplotlib.pyplot as plt

for name, km in cluster_models.items():
    centroids_scaled, plot_feat_names = get_centroids_scaled(km)
    n_clusters = centroids_scaled.shape[0]
    n_feats = centroids_scaled.shape[1]
    
    # create figure
    fig, ax = plt.subplots(figsize=(14, max(8, n_clusters * 0.4)))
    
    # plot each cluster as a row
    for i in range(n_clusters):
        values = centroids_scaled[i]
        # scatter with y-position = cluster id
        ax.scatter(np.arange(n_feats), np.full(n_feats, i), 
                  s=values * 200,  # size proportional to scaled value
                  c=values, cmap='RdYlBu_r', vmin=0, vmax=1,
                  alpha=0.7, edgecolors='black', linewidths=0.5)
    
    # formatting
    ax.set_yticks(np.arange(n_clusters))
    ax.set_yticklabels([f'cluster {i}' for i in range(n_clusters)])
    ax.set_xticks(np.arange(n_feats))
    ax.set_xticklabels(plot_feat_names, rotation=45, ha='right', fontsize=9)
    ax.set_xlabel('Feature', fontsize=12)
    ax.set_ylabel('Cluster', fontsize=12)
    ax.set_title(f'Cluster centroids (dot size = scaled value) — {name}', fontsize=14)
    ax.grid(axis='x', alpha=0.3)
    
    # colorbar
    sm = plt.cm.ScalarMappable(cmap='RdYlBu_r', norm=plt.Normalize(vmin=0, vmax=1))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label('Scaled value (0-1)', fontsize=10)
    
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np

# Dot plot of cluster centroids for each feature (min-max scaled per feature, like cell 47)
# Uses existing cluster_models dict (km_best, km_15)

import matplotlib.pyplot as plt

for name, km in cluster_models.items():
    # get centroids in original scale
    cent = km.cluster_centers_  # in scaled space used by clustering
    # inverse-transform with scaler to original scale
    if 'scaler' in globals() and hasattr(scaler, "inverse_transform"):
        try:
            cent_orig = scaler.inverse_transform(cent)
        except Exception:
            cent_orig = cent
    else:
        cent_orig = cent
    
    # get feature names (same as used for clustering)
    if 'X' in globals() and hasattr(X, "columns"):
        feat_names = list(X.columns)
    else:
        feat_names = list(df.select_dtypes(include=[np.number]).drop(columns=['label','which_motif'], errors='ignore').columns)
    
    # align number of features between centroids and feat_names
    if cent_orig.shape[1] != len(feat_names):
        n_use = min(cent_orig.shape[1], len(feat_names))
        cent_orig = cent_orig[:, :n_use]
        feat_names = feat_names[:n_use]
    
    # min-max scale each feature to [0, 1] for comparable marker size/color
    feat_min = cent_orig.min(axis=0)
    feat_max = cent_orig.max(axis=0)
    denom = feat_max - feat_min
    denom[denom == 0] = 1.0
    scaled_vals = (cent_orig - feat_min) / denom
    
    n_clusters = scaled_vals.shape[0]
    n_feats = scaled_vals.shape[1]
    
    # create figure
    fig, ax = plt.subplots(figsize=(14, max(8, n_clusters * 0.4)))
    
    # plot each cluster as a row
    for i in range(n_clusters):
        values = scaled_vals[i]
        # scatter with y-position = cluster id
        ax.scatter(np.arange(n_feats), np.full(n_feats, i), 
                  s=values * 200,  # size proportional to scaled value
                  c=values, cmap='RdYlBu_r', vmin=0, vmax=1,
                  alpha=0.7, edgecolors='black', linewidths=0.5)
    
    # formatting
    ax.set_yticks(np.arange(n_clusters))
    ax.set_yticklabels([f'cluster {i}' for i in range(n_clusters)])
    ax.set_xticks(np.arange(n_feats))
    ax.set_xticklabels(feat_names, rotation=45, ha='right', fontsize=9)
    ax.set_xlabel('Feature', fontsize=12)
    ax.set_ylabel('Cluster', fontsize=12)
    ax.set_title(f'Cluster centroids (dot size = scaled value, per-feature min-max) — {name}', fontsize=14)
    ax.grid(axis='x', alpha=0.3)
    
    # colorbar
    sm = plt.cm.ScalarMappable(cmap='RdYlBu_r', norm=plt.Normalize(vmin=0, vmax=1))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label('Scaled value (0-1)', fontsize=10)
    
    plt.tight_layout()
    plt.show()

In [ ]:

cell_type_protein = melanoma_protein[wanted_columns + ['pred']].drop(['centroid-0','centroid-1','fov','patient number'], axis=1)    # create figure



In [ ]:
translator = {}
for cell_type in melanoma_protein['pred'].unique():
    
    cell_type = cell_type.strip().lower()

    if 'mac' in cell_type or 'mono_cd14_dr' == cell_type:
        translator[cell_type] = 'mac'
        
    elif 'collagen' in cell_type or 'sma' in cell_type:
        translator[cell_type] = 'stroma'
    
    elif 'cd45ra+mhc2+' in cell_type:
        translator[cell_type] = 'b cell'
    
    elif cell_type in ['cd14_cd11c_dcs','cd11_cd11c_dcsign_dcs', 'dcs']:
        translator[cell_type] = 'DC'
        
    elif cell_type == 'tfh%cd4 t cell':
        translator[cell_type] = 'cd4 t cell'


    elif cell_type == 'cd4 treg%cd4 t cell':
        translator[cell_type] = 'treg'
    
    elif cell_type == 'follicular_germinal_b_cell':
        translator[cell_type] = 'gc'
        
    else:
        translator[cell_type] = cell_type

translator

In [ ]:
cell_type_protein['pred'] = cell_type_protein['pred'].apply(lambda x: translator[x.strip().lower()])
cell_type_protein = cell_type_protein.groupby(['pred']).mean().reset_index()

In [ ]:
cell_type_protein['pred'].unique()

In [ ]:
cell_type_protein.to_csv('cell_type_protein.csv')

In [ ]:
# Dot plot of mean protein expression per predicted cell type (min-max scaled per feature)

features = [c for c in cell_type_protein.columns if c != 'pred']
df_plot = cell_type_protein.copy()

# min–max scale each feature to [0, 1] for comparable marker size/color
feat_min = df_plot[features].min()
feat_max = df_plot[features].max()
denom = feat_max - feat_min
denom[denom == 0] = 1.0
scaled_vals = (df_plot[features] - feat_min) / denom

n_clusters = df_plot.shape[0]
n_feats = len(features)
fig, ax = plt.subplots(figsize=(14, max(8, n_clusters * 0.4)))

scaled_vals.to_csv('scaled_vals.csv')

for i in range(n_clusters):
    values = scaled_vals.iloc[i].values
    ax.scatter(
        np.arange(n_feats),
        np.full(n_feats, i),
        s=values * 200,
        c=values,
        cmap='RdYlBu_r',
        vmin=0,
        vmax=1,
        alpha=0.7,
        edgecolors='black',
        linewidths=0.5,
    )

ax.set_yticks(np.arange(n_clusters))
ax.set_yticklabels(df_plot['pred'])
ax.set_xticks(np.arange(n_feats))
ax.set_xticklabels(features, rotation=45, ha='right', fontsize=9)
ax.set_xlabel('Feature', fontsize=12)
ax.set_ylabel('Pred cell type', fontsize=12)
ax.set_title('Mean protein expression per predicted cell type (scaled)', fontsize=14)
ax.grid(axis='x', alpha=0.3)

sm = plt.cm.ScalarMappable(cmap='RdYlBu_r', norm=plt.Normalize(vmin=0, vmax=1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label('Scaled value (0–1)', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
df_in_motif = df[df['label'] == 1]
df_in_motif.head(5)

In [ ]:
df_in_motif['which_motif'].value_counts()

In [ ]:
motif_class_mapper = {20748339660: 'NN',16417793484:'NP', 17763644908:'NN',4880694732:'NP',15341317580:'NP',4071751116:'NP',3522784716:'NP'}
df_in_motif['class'] = df_in_motif['which_motif'].map(motif_class_mapper)
df_in_motif.head(5)

In [ ]:
df_in_motif.drop(columns=['cluster_best','cluster_15','label'], inplace=True)

In [ ]:
# Random forest to predict 'class' in df_in_motif
# Re-uses sklearn, numpy, pandas objects already in the notebook namespace.
from random import shuffle
# Prepare data
df_model = df_in_motif.dropna(subset=['class']).copy()
# map classes to binary labels
label_map = {'NN': 0, 'NP': 1}
X = df_model.select_dtypes(include=[np.number]).drop(columns=['which_motif','cluster_best','cluster_15','class'], errors='ignore')
accuracies = []

for trial in range(100):
    # features: numeric columns only, exclude which_motif if present
    y = df_model['class'].map(label_map)
    y = y.sample(frac=1, random_state=trial).reset_index(drop=True)
    # if there are any non-informative columns like patient/fov present as numeric, drop them
    for c in ['patient number', 'fov','centroid-0', 'centroid-1']:
        if c in X.columns:
            X = X.drop(columns=[c])

    # check there is data
    if X.shape[0] == 0 or y.nunique() < 2:
        raise RuntimeError("Insufficient data or no class variability to train classifier.")

    # train/test split with stratify when possible
    rs = 42069
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=rs, stratify=y)
    except ValueError:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=rs, stratify=None)

    # fit random forest
    rf_motif = RandomForestClassifier(n_estimators=200, random_state=rs, n_jobs=-1)
    rf_motif.fit(X_train, y_train)

    # evaluate
    y_pred = rf_motif.predict(X_test)
    print("Accuracy:", metrics.accuracy_score(y_test, y_pred))
    print(metrics.classification_report(y_test, y_pred, digits=4))
    accuracies.append(metrics.accuracy_score(y_test, y_pred))
    # confusion matrix
    cm = metrics.confusion_matrix(y_test, y_pred)
    print("Confusion matrix:\n", cm)

    # ROC AUC if possible
    if len(np.unique(y_test)) == 2:
        try:
            y_proba = rf_motif.predict_proba(X_test)[:, 1]
            auc = metrics.roc_auc_score(y_test, y_proba)
            print(f"ROC AUC: {auc:.4f}")
        except Exception:
            pass

# feature importances (top 20)
importances = pd.Series(rf_motif.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top features:\n", importances.head(20))

In [ ]:
from scipy import stats
import numpy as np
mean_accuracy = np.mean(accuracies)
std_accuracy = np.std(accuracies)

print(f"Mean accuracy: {mean_accuracy:.4f}")
print(f"Standard deviation: {std_accuracy:.4f}")
print(f"Accuracy: {mean_accuracy:.4f} ± {std_accuracy:.4f}")


# t-test: null hypothesis that mean accuracy equals 0.85
t_stat, p_value = stats.ttest_1samp(accuracies, 0.85)
print(f"\nt-test (H0: mean accuracy = 0.85):")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: Reject H0 (accuracy is significantly different from 0.85)")
else:
    print("Result: Fail to reject H0 (accuracy is not significantly different from 0.85)")

In [ ]:



# Keep the trained model and train/test splits in the notebook namespace
# rf_motif, X_train, X_test, y_train, y_test are available for further inspection.

# Cell-type association probability for in-motif unidentified cells

For each unidentified cell that participates in a motif, compute its Euclidean distance to the mean
protein-expression vector of every known cell type (both scaled the same way).  
Two probability versions are produced:
- **Inverse-distance** — `p_i = (1/d_i) / Σ(1/d_j)` (raw proportion)
- **Softmax** — `p_i = exp(−d_i) / Σ exp(−d_j)` (temperature-free, sharper)

Probabilities are then averaged per motif and shown as a heatmap (motifs × cell types).


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cdist

# ---------------------------------------------------------------
# 1.  Align features between df_in_motif and cell_type_protein
# ---------------------------------------------------------------
# protein features present in cell_type_protein (excluding 'pred')
ct_features = [c for c in cell_type_protein.columns if c != 'pred']

# features that also exist in df_in_motif (numeric, excluding bookkeeping cols)
non_feature_cols = {'label', 'which_motif', 'cluster_best', 'cluster_8', 'cluster_15', 'class'}
cell_features = [c for c in df_in_motif.columns
                 if c in ct_features and c not in non_feature_cols]

print(f"Using {len(cell_features)} shared protein features.")

# ---------------------------------------------------------------
# 2.  Min-max scale unidentified cells with the SAME feat_min/feat_max
#     that was used to build scaled_vals in the dot-plot cell
# ---------------------------------------------------------------
# feat_min / feat_max are Series indexed by feature name (from cell 54)
shared_min = feat_min.reindex(cell_features)
shared_max = feat_max.reindex(cell_features)
shared_denom = (shared_max - shared_min).replace(0, 1.0)

cells_raw = df_in_motif[cell_features].copy()
cells_scaled = (cells_raw - shared_min) / shared_denom   # shape: (n_cells, n_feats)

# reference centroids (already min-max scaled with the same parameters)
# scaled_vals rows correspond to cell_type_protein rows
ct_names = cell_type_protein['pred'].tolist()
# align column order
ref_matrix = scaled_vals[cell_features].values          # shape: (n_cell_types, n_feats)

# ---------------------------------------------------------------
# 3.  Compute per-cell Euclidean distances to each cell-type centroid
# ---------------------------------------------------------------
cells_arr = cells_scaled.values                          # (n_cells, n_feats)
dist_matrix = cdist(cells_arr, ref_matrix, metric='euclidean')  # (n_cells, n_cell_types)

# ---------------------------------------------------------------
# 4a. Inverse-distance probability  p_i = (1/d_i) / Σ(1/d_j)
# ---------------------------------------------------------------
eps = 1e-9                                               # avoid division by zero
inv_dist = 1.0 / (dist_matrix + eps)
prob_inv = inv_dist / inv_dist.sum(axis=1, keepdims=True)

# ---------------------------------------------------------------
# 4b. Softmax probability  p_i = exp(-d_i) / Σ exp(-d_j)
# ---------------------------------------------------------------
neg_dist = -dist_matrix
neg_dist_stable = neg_dist - neg_dist.max(axis=1, keepdims=True)  # numerical stability
exp_nd = np.exp(neg_dist_stable)
prob_softmax = exp_nd / exp_nd.sum(axis=1, keepdims=True)

# ---------------------------------------------------------------
# 5.  Wrap into DataFrames and attach motif labels
# ---------------------------------------------------------------
prob_inv_df = pd.DataFrame(prob_inv, index=df_in_motif.index, columns=ct_names)
prob_softmax_df = pd.DataFrame(prob_softmax, index=df_in_motif.index, columns=ct_names)
prob_inv_df['which_motif'] = df_in_motif['which_motif'].values
prob_softmax_df['which_motif'] = df_in_motif['which_motif'].values

# ---------------------------------------------------------------
# 6.  Average per motif
# ---------------------------------------------------------------
motif_avg_inv     = prob_inv_df.groupby('which_motif')[ct_names].mean()
motif_avg_softmax = prob_softmax_df.groupby('which_motif')[ct_names].mean()

# sort motifs for readability
motif_avg_inv     = motif_avg_inv.sort_index()
motif_avg_softmax = motif_avg_softmax.sort_index()

print("Motif × cell-type probability matrices (inverse-distance and softmax):")
print(f"  Shape: {motif_avg_inv.shape}")

# ---------------------------------------------------------------
# 7.  Heatmaps
# ---------------------------------------------------------------
def _prob_heatmap(data: pd.DataFrame, title: str, ax):
    sns.heatmap(
        data, annot=True, fmt='.2f', cmap='YlOrRd',
        vmin=0, vmax=data.values.max(),
        ax=ax, linewidths=0.3,
        cbar_kws={'label': 'Avg probability'}
    )
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Cell type', fontsize=10)
    ax.set_ylabel('Motif', fontsize=10)
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)

n_motifs_plot = motif_avg_inv.shape[0]
n_ct_plot     = motif_avg_inv.shape[1]
fig_w = max(12, n_ct_plot * 0.7 + 3)
fig_h = max(5, n_motifs_plot * 0.55 + 2)

fig, axes = plt.subplots(1, 2, figsize=(fig_w * 2, fig_h))
_prob_heatmap(motif_avg_inv,     'Cell-type association (inverse-distance)', axes[0])
_prob_heatmap(motif_avg_softmax, 'Cell-type association (softmax)',          axes[1])

plt.suptitle('Average cell-type association probability per motif\n(in-motif unidentified cells)',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


## Spectrogram: all unidentified cells × protein expression (ordered by cluster)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# ── 1. Choose cluster column ──────────────────────────────────────────────────
cluster_col = 'cluster_15' if 'cluster_15' in df.columns else cluster_cols[0]

# ── 2. Protein feature columns ────────────────────────────────────────────────
non_feat = {'label', 'which_motif', 'cluster_best', 'cluster_8', 'cluster_15', 'class'}
feat_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in non_feat]

# ── 3. Build working frame: protein features + cluster label ──────────────────
work = df[feat_cols + [cluster_col, 'label']].dropna(subset=[cluster_col]).copy()
work[cluster_col] = work[cluster_col].astype(int)

# sort rows by hierarchical cluster order
ordered_clusters = hierarchical_cluster_order(work, cluster_col, feature_cols=feat_cols)
if not ordered_clusters:
    ordered_clusters = sorted(work[cluster_col].unique().tolist())
cluster_rank = {c: i for i, c in enumerate(ordered_clusters)}
work = work.assign(_cluster_rank=work[cluster_col].map(cluster_rank)).sort_values(['_cluster_rank', cluster_col]).drop(columns=['_cluster_rank']).reset_index(drop=True)

# ── 4. Center each protein by mean across all unidentified cells ──────────────
X_raw = work[feat_cols].copy().astype(float)
unidentified_mean = X_raw.mean(axis=0)
X_centered = X_raw - unidentified_mean

# Scale centered values to [-1, 1] for diverging visualization
abs_max = np.abs(X_centered.values).max()
if abs_max == 0:
    abs_max = 1.0
X_hm = X_centered / abs_max

print('Baseline correction: each protein value shown as (cell - mean across all unidentified cells).')
print(f'Centered value scale: [{X_hm.values.min():.3f}, {X_hm.values.max():.3f}]')

# ── 5. Build cluster colour bar ───────────────────────────────────────────────
unique_clusters = ordered_clusters
n_clust = len(unique_clusters)
cmap_clust  = plt.colormaps.get_cmap('tab20').resampled(max(n_clust, 2))
clust_color = {c: cmap_clust(i) for i, c in enumerate(unique_clusters)}
row_colors  = np.array([clust_color[c] for c in work[cluster_col]])  # (n_cells, 4)

# ── 6. Figure layout — fixed PowerPoint-friendly size (16:9) ─────────────────
n_cells, n_feats = X_hm.shape
fig_w, fig_h = 13.33, 6.5   # standard 16:9 slide dimensions in inches

fig = plt.figure(figsize=(fig_w, fig_h))
gs  = GridSpec(
    1, 3,
    width_ratios=[0.018, 1, 0.025],   # cluster bar | heatmap | colorbar
    wspace=0.02,
    left=0.05, right=0.92, top=0.90, bottom=0.22
)

# ── 7. Left strip: cluster identity ──────────────────────────────────────────
ax_strip = fig.add_subplot(gs[0, 0])
strip_img = row_colors[:, :3][np.newaxis, :, :].transpose(1, 0, 2)  # (n_cells,1,3)
ax_strip.imshow(strip_img, aspect='auto', interpolation='none')
ax_strip.set_xticks([])
ax_strip.set_yticks([])
ax_strip.set_ylabel('Cells (sorted by cluster)', fontsize=9)

# ── 8. Main heatmap ───────────────────────────────────────────────────────────
ax_hm = fig.add_subplot(gs[0, 1])
im = ax_hm.imshow(
    X_hm.values,
    aspect='auto',
    interpolation='none',
    cmap='RdBu_r',
    vmin=-1, vmax=1,
    origin='upper'
)
ax_hm.set_xticks(np.arange(n_feats))
ax_hm.set_xticklabels(feat_cols, rotation=60, ha='right', fontsize=7)
ax_hm.set_yticks([])
ax_hm.set_xlabel('Protein', fontsize=10)
ax_hm.set_title(
    f'Unidentified cells — protein expression spectrogram (ordered by {cluster_col}, centered vs unidentified mean)',
    fontsize=11, pad=6
)

# draw cluster boundary lines
boundaries = np.where(np.diff(work[cluster_col].values))[0] + 0.5
for b in boundaries:
    ax_hm.axhline(b, color='black', linewidth=0.3, alpha=0.4)

# ── 9. Colour bar (expression scale) ─────────────────────────────────────────
ax_cb = fig.add_subplot(gs[0, 2])
plt.colorbar(im, cax=ax_cb, label='Centered expression (scaled: -1 to 1)')

# ── 10. Cluster legend ────────────────────────────────────────────────────────
patches = [mpatches.Patch(color=clust_color[c], label=f'cluster {c}') for c in unique_clusters]
fig.legend(
    handles=patches,
    loc='lower center',
    ncol=min(n_clust, 10),
    fontsize=7,
    title=cluster_col,
    title_fontsize=8,
    bbox_to_anchor=(0.5, 0.0),
    frameon=True
)

plt.show()
print(f'Spectrogram: {n_cells} cells × {n_feats} proteins, {n_clust} clusters.')

In [ ]:
protein

In [ ]:
melanoma_protein_unidentified.head(5)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from sklearn.neighbors import NearestNeighbors

# ── 1. Prepare data: use melanoma_cells (compact dataset) + filter to NN, NP ────
all_cells = melanoma_cells[melanoma_cells['Group'].isin(['NN', 'NP'])].copy()

print(f"Total cells (NN & NP groups): {len(all_cells)}")
print(f"Number of FOVs: {all_cells['fov'].nunique()}")

# ── 2. Compute nearest neighbor distance within each FOV ──────────────────────
# Initialize nn_distance column
all_cells['nn_distance'] = np.nan

fov_list = all_cells['fov'].unique()
for fov in fov_list:
    # Get cells in this FOV
    fov_mask = all_cells['fov'] == fov
    fov_indices = np.where(fov_mask)[0]
    
    # Extract spatial coordinates for this FOV
    fov_coords = all_cells.loc[fov_mask, ['centroid-0', 'centroid-1']].values
    
    if len(fov_coords) > 1:
        # Compute nearest neighbors within this FOV
        nbrs = NearestNeighbors(n_neighbors=2, algorithm='ball_tree').fit(fov_coords)
        distances, indices = nbrs.kneighbors(fov_coords)
        
        # distances[:, 1] is the distance to nearest neighbor
        all_cells.loc[fov_mask, 'nn_distance'] = distances[:, 1]
    else:
        # FOV has only 1 cell, nearest neighbor distance is NaN
        all_cells.loc[fov_mask, 'nn_distance'] = np.nan

print(f"\nNearest neighbor distances computed (within FOV):")
print(f"  Cells with valid nn_distance: {all_cells['nn_distance'].notna().sum()}")
print(f"  Min distance: {all_cells['nn_distance'].min():.4f}")
print(f"  Max distance: {all_cells['nn_distance'].max():.4f}")
print(f"  Mean distance: {all_cells['nn_distance'].mean():.4f}")
print(f"  Median distance: {all_cells['nn_distance'].median():.4f}")

# ── 3. Add label and which_motif columns from melanoma_protein ──────────────────
# Get prediction, label, and which_motif columns from melanoma_protein
merge_keys = ['patient number', 'fov', 'cell_id']
label_info = melanoma_protein[merge_keys + ['label', 'which_motif']].drop_duplicates()

all_cells = all_cells.merge(
    label_info, 
    on=merge_keys, 
    how='left'
)

# Fill NaN labels with 0 (not in-motif)
all_cells['label'] = all_cells['label'].fillna(0).astype(int)
all_cells['which_motif'] = all_cells['which_motif'].fillna(0).astype(int)

print(f"\nAfter adding labels and motif info:")
print(f"  Cells with label=1: {(all_cells['label'] == 1).sum()}")
print(f"  Motifs present: {sorted([m for m in all_cells['which_motif'].unique() if m != 0])}")

# ── 4. Categorize cells ──────────────────────────────────────────────────────
# Category 1: All cells
all_cells['category'] = 'all'

# Category 2: All unidentified cells
all_cells.loc[all_cells['pred'] == 'Unidentified', 'category'] = 'unidentified'

# Category 3: In-motif unidentified cells
all_cells.loc[(all_cells['pred'] == 'Unidentified') & (all_cells['label'] == 1), 'category'] = 'in-motif'

# Print category counts
print(f"\nCell categories (NN & NP):")
for cat in ['all', 'unidentified', 'in-motif']:
    count = len(all_cells[all_cells['category'] == cat])
    print(f"  {cat}: {count}")

# ── 5. Create overall visualization: histograms + KDE distribution plots ──────
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Define colors for each category
colors = {'all': 'steelblue', 'unidentified': 'orange', 'in-motif': 'red'}
categories = ['all', 'unidentified', 'in-motif']

# Filter out NaN distances for visualization
valid_distances = all_cells['nn_distance'].dropna()

# ── Subplot 1: Overlaid histograms ────────────────────────────────────────────
ax1 = axes[0]
bins = np.linspace(valid_distances.min(), valid_distances.max(), 50)

for cat in categories:
    cat_data = all_cells[all_cells['category'] == cat]['nn_distance'].dropna()
    if len(cat_data) > 0:
        ax1.hist(cat_data, bins=bins, alpha=0.5, label=f'{cat} (n={len(cat_data)})', 
                 color=colors[cat], edgecolor='black', linewidth=0.5)

ax1.set_xlabel('Distance to nearest neighbor (within FOV)', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title('Histogram: Spatial Distance to Nearest Neighbor by Cell Category (NN & NP)', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(axis='y', alpha=0.3)

# ── Subplot 2: KDE distribution plots ─────────────────────────────────────────
ax2 = axes[1]

for cat in categories:
    cat_data = all_cells[all_cells['category'] == cat]['nn_distance'].dropna().values
    
    if len(cat_data) > 1:
        # Compute KDE
        kde = gaussian_kde(cat_data)
        x_range = np.linspace(valid_distances.min(), valid_distances.max(), 200)
        y_kde = kde(x_range)
        
        ax2.plot(x_range, y_kde, linewidth=2.5, label=f'{cat} (n={len(cat_data)})', 
                color=colors[cat])
        ax2.fill_between(x_range, y_kde, alpha=0.2, color=colors[cat])

ax2.set_xlabel('Distance to nearest neighbor (within FOV)', fontsize=12)
ax2.set_ylabel('Density', fontsize=12)
ax2.set_title('KDE Distribution: Spatial Distance to Nearest Neighbor by Cell Category (NN & NP)', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ── 6. Summary statistics ────────────────────────────────────────────────────
print("\n" + "="*70)
print("SUMMARY STATISTICS: Spatial Distance to Nearest Neighbor (NN & NP)")
print("="*70)

for cat in categories:
    cat_data = all_cells[all_cells['category'] == cat]['nn_distance'].dropna()
    if len(cat_data) > 0:
        print(f"\n{cat.upper()}  (n={len(cat_data)}):")
        print(f"  Min:    {cat_data.min():.4f}")
        print(f"  Q1:     {cat_data.quantile(0.25):.4f}")
        print(f"  Median: {cat_data.median():.4f}")
        print(f"  Q3:     {cat_data.quantile(0.75):.4f}")
        print(f"  Max:    {cat_data.max():.4f}")
        print(f"  Mean:   {cat_data.mean():.4f}")
        print(f"  Std:    {cat_data.std():.4f}")

# ── 7. Per-motif distribution plots ───────────────────────────────────────────
# Get in-motif unidentified cells with valid motif information
in_motif_cells = all_cells[(all_cells['category'] == 'in-motif') & (all_cells['which_motif'] != 0)]
motif_list = sorted(in_motif_cells['which_motif'].unique())

if len(motif_list) > 0:
    print(f"\n" + "="*70)
    print(f"PER-MOTIF ANALYSIS: {len(motif_list)} motifs detected")
    print("="*70)
    
    # Create subplots with 2 rows per motif (histogram + KDE)
    n_motifs = len(motif_list)
    fig, axes = plt.subplots(n_motifs, 2, figsize=(16, 4.5 * n_motifs))
    
    # Handle case where there's only 1 motif
    if n_motifs == 1:
        axes = axes.reshape(1, -1)
    
    for idx, motif in enumerate(motif_list):
        motif_data = in_motif_cells[in_motif_cells['which_motif'] == motif]['nn_distance'].dropna()
        all_in_motif_data = in_motif_cells['nn_distance'].dropna()
        
        # ── Histogram ────────────────────────────────────────────────────────
        ax_hist = axes[idx, 0]
        bins = np.linspace(all_in_motif_data.min(), all_in_motif_data.max(), 30)
        ax_hist.hist(motif_data, bins=bins, alpha=0.7, color='red', edgecolor='black', linewidth=0.8)
        ax_hist.set_xlabel('Distance to nearest neighbor', fontsize=11)
        ax_hist.set_ylabel('Count', fontsize=11)
        ax_hist.set_title(f'Motif {motif}: Histogram (n={len(motif_data)})', fontsize=12, fontweight='bold')
        ax_hist.grid(axis='y', alpha=0.3)
        
        # Add statistics text
        stats_text = f"μ={motif_data.mean():.2f}\nσ={motif_data.std():.2f}\nMed={motif_data.median():.2f}"
        ax_hist.text(0.98, 0.97, stats_text, transform=ax_hist.transAxes,
                    fontsize=9, verticalalignment='top', horizontalalignment='right',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        # ── KDE plot ─────────────────────────────────────────────────────────
        ax_kde = axes[idx, 1]
        if len(motif_data) > 1:
            kde = gaussian_kde(motif_data)
            x_range = np.linspace(all_in_motif_data.min(), all_in_motif_data.max(), 200)
            y_kde = kde(x_range)
            
            ax_kde.plot(x_range, y_kde, linewidth=2.5, color='red')
            ax_kde.fill_between(x_range, y_kde, alpha=0.3, color='red')
        ax_kde.set_xlabel('Distance to nearest neighbor', fontsize=11)
        ax_kde.set_ylabel('Density', fontsize=11)
        ax_kde.set_title(f'Motif {motif}: KDE Distribution (n={len(motif_data)})', fontsize=12, fontweight='bold')
        ax_kde.grid(alpha=0.3)
        
        # Print statistics for this motif
        print(f"\nMotif {motif} (n={len(motif_data)}):")
        print(f"  Min:    {motif_data.min():.4f}")
        print(f"  Q1:     {motif_data.quantile(0.25):.4f}")
        print(f"  Median: {motif_data.median():.4f}")
        print(f"  Q3:     {motif_data.quantile(0.75):.4f}")
        print(f"  Max:    {motif_data.max():.4f}")
        print(f"  Mean:   {motif_data.mean():.4f}")
        print(f"  Std:    {motif_data.std():.4f}")
    
    plt.tight_layout()
    plt.show()
else:
    print("\nNo in-motif cells found with valid motif information.")

## Spectrogram: macrophage cells by subtype (subtype-as-cluster)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# User-provided mapping from raw pred labels to canonical classes
pred_map = {
    'unidentified': 'unidentified',
    'memory%cd4 t cell': 'memory%cd4 t cell',
    'hevs': 'hevs',
    'cd8 t cell': 'cd8 t cell',
    'cd4 t cell': 'cd4 t cell',
    'cd45ra+mhc2+': 'b cell',
    'nk cell': 'nk cell',
    'mac': 'mac',
    'blood vessels': 'blood vessels',
    'sma': 'stroma',
    'cd4 apc': 'cd4 apc',
    'b cell': 'b cell',
    'dcs': 'DC',
    'collagen_sma': 'stroma',
    'mono_cd14_dr': 'mac',
    'collagen': 'stroma',
    'neutrophil': 'neutrophil',
    'cd4 treg%cd4 t cell': 'treg',
    'dc sign mac': 'mac',
    'cd206_mac': 'mac',
    'follicular_germinal_b_cell': 'gc',
    'tfh%cd4 t cell': 'cd4 t cell',
    'cd14_cd11c_dcs': 'DC',
    'cd68_mac': 'mac',
    'cd11_cd11c_dcsign_dcs': 'DC',
    'tumor': 'tumor'
}

# Build a set of raw subtype names that map to macrophages
mac_subtype_keys = {k for k, v in pred_map.items() if str(v).strip().lower() == 'mac'}

# Use protein-level table (already filtered to NN/NP earlier in the notebook)
macrophage_df = melanoma_protein.copy()
macrophage_df['pred_clean'] = macrophage_df['pred'].astype(str).str.strip().str.lower()
macrophage_df = macrophage_df[macrophage_df['pred_clean'].isin(mac_subtype_keys)].copy()

if macrophage_df.empty:
    raise RuntimeError('No macrophage cells found with the provided pred_map.')

# Each subtype is a cluster label (no unsupervised clustering)
macrophage_df['mac_subtype'] = macrophage_df['pred_clean']

# Protein feature columns (exclude coordinates/metadata)
feat_cols = [
    c for c in wanted_columns
    if c in macrophage_df.columns and c not in {'patient number', 'fov', 'centroid-0', 'centroid-1'}
]

# Keep rows with all selected protein features
work = macrophage_df[['mac_subtype'] + feat_cols].dropna().copy()

# Order subtype-clusters by abundance
subtype_counts = work['mac_subtype'].value_counts()
ordered_subtypes = subtype_counts.index.tolist()
work['mac_subtype'] = pd.Categorical(work['mac_subtype'], categories=ordered_subtypes, ordered=True)
work = work.sort_values('mac_subtype').reset_index(drop=True)

# Subtract average protein expression across all macrophage cells
mac_cells_ref = macrophage_df[feat_cols].dropna().astype(float)
global_mean = mac_cells_ref.mean(axis=0)
X_centered = work[feat_cols].astype(float) - global_mean

# Scale centered values to [-1, 1] for diverging visualization
abs_max = np.abs(X_centered.values).max()
if abs_max == 0:
    abs_max = 1.0
X_hm = X_centered / abs_max

# Row color strip by subtype
n_subtypes = len(ordered_subtypes)
cmap_sub = plt.colormaps.get_cmap('tab20').resampled(max(2, n_subtypes))
sub_color = {s: cmap_sub(i) for i, s in enumerate(ordered_subtypes)}
row_colors = np.array([sub_color[s] for s in work['mac_subtype']])

# Print summary
print(f'Macrophage cells used: {len(work)}')
print(f'Macrophage subtype-clusters: {n_subtypes}')
print('Subtype counts:')
for s in ordered_subtypes:
    print(f'  {s}: {int(subtype_counts[s])}')
print('\nBaseline correction: each protein value shown as (cell - mean across macrophage cells).')
print(f'Centered value scale: [{X_hm.values.min():.3f}, {X_hm.values.max():.3f}]')

# Figure layout (same style as unidentified spectrogram)
n_cells, n_feats = X_hm.shape
fig_w, fig_h = 13.33, 6.5

fig = plt.figure(figsize=(fig_w, fig_h))
gs = GridSpec(
    1, 3,
    width_ratios=[0.018, 1, 0.025],
    wspace=0.02,
    left=0.05, right=0.92, top=0.90, bottom=0.25
)

# Left strip: subtype identity
ax_strip = fig.add_subplot(gs[0, 0])
strip_img = row_colors[:, :3][np.newaxis, :, :].transpose(1, 0, 2)
ax_strip.imshow(strip_img, aspect='auto', interpolation='none')
ax_strip.set_xticks([])
ax_strip.set_yticks([])
ax_strip.set_ylabel('Cells (grouped by macrophage subtype)', fontsize=9)

# Main heatmap
ax_hm = fig.add_subplot(gs[0, 1])
im = ax_hm.imshow(
    X_hm.values,
    aspect='auto',
    interpolation='none',
    cmap='RdBu_r',
    vmin=-1, vmax=1,
    origin='upper'
)
ax_hm.set_xticks(np.arange(n_feats))
ax_hm.set_xticklabels(feat_cols, rotation=60, ha='right', fontsize=7)
ax_hm.set_yticks([])
ax_hm.set_xlabel('Protein', fontsize=10)
ax_hm.set_title('Macrophage cells - protein expression spectrogram (subtype as cluster, centered vs macrophage mean)', fontsize=11, pad=6)

# Draw subtype boundary lines
sub_codes = work['mac_subtype'].cat.codes.values
boundaries = np.where(np.diff(sub_codes))[0] + 0.5
for b in boundaries:
    ax_hm.axhline(b, color='black', linewidth=0.3, alpha=0.4)

# Colorbar
ax_cb = fig.add_subplot(gs[0, 2])
plt.colorbar(im, cax=ax_cb, label='Centered expression (scaled: -1 to 1)')

# Legend
patches = [
    mpatches.Patch(color=sub_color[s], label=f'{s} (n={int(subtype_counts[s])})')
    for s in ordered_subtypes
]
fig.legend(
    handles=patches,
    loc='lower center',
    ncol=min(n_subtypes, 4),
    fontsize=8,
    title='Macrophage subtype (cluster)',
    title_fontsize=9,
    bbox_to_anchor=(0.5, 0.0),
    frameon=True
)

plt.show()
print(f'Spectrogram: {n_cells} macrophage cells x {n_feats} proteins, {n_subtypes} subtype-clusters.')